# Base of Three — a tiny survivor game on the NPC engine

Three survivors, three days. Assign tasks each morning; every evening each survivor says a line generated from their live state (morale, injuries, infection). Keep everyone alive, fed, and from giving up until day 4.

**Runtime must be a T4 GPU.** Run the cells top to bottom, then type your task choices (scavenge / guard / rest) when asked.

## 1. Install (pinned — Dream's code breaks on newer transformers)

In [ ]:
!pip install -q transformers==4.53.3 accelerate bitsandbytes

## 2. Restore the npc_engine package

In [ ]:
# --- restore the npc_engine package (embedded in this notebook) ---
import base64, pathlib
FILES = {
    'npc_engine/__init__.py': 'IiIiTlBDIEVuZ2luZSBQaGFzZSAwIOKAlCBhIHRpbnkgdGV4dC1kaWZmdXNpb24gTlBDIGRpYWxvZ3VlIGhhcm5lc3MuCgpCdWlsdCBvbiBEcmVhbSA3QiBJbnN0cnVjdCAoRHJlYW0tb3JnL0RyZWFtLXYwLUluc3RydWN0LTdCKSwgYSBtYXNrZWQgZGlmZnVzaW9uIExNLgpUaGUgZGlmZnVzaW9uLW5hdGl2ZSBmZWF0dXJlcyB1c2VkIGhlcmU6CgotIHBhcmFsbGVsIGNhbmRpZGF0ZSBnZW5lcmF0aW9uOiBOIG1hc2tlZCBjYW52YXNlcyBkZW5vaXNlZCBpbiBPTkUgYmF0Y2hlZCBwYXNzCiAgKEFSIHdvdWxkIG5lZWQgTiBzZXF1ZW50aWFsIGRlY29kZXMpCi0gY29uZmlkZW5jZSByYW5raW5nOiBwZXItdG9rZW4gbWF4LXByb2Igb3ZlciB0aGUgZ2VuZXJhdGVkIHNwYW4sIGZyb20gYSBzaW5nbGUKICBzY29yaW5nIGZvcndhcmQgcGFzcyDigJQgbm8gc2Vjb25kIG1vZGVsCi0gY29uZmlkZW5jZSBmYWxsYmFjazogaWYgdGhlIGJlc3QgY2FuZGlkYXRlIHNjb3JlcyBiZWxvdyB0aHJlc2hvbGQsIHRoZSBlbmdpbmUKICBzaGlwcyBhIHNhZmUgY2FubmVkIGxpbmUgaW5zdGVhZCBvZiBhIGJyb2tlbiBvbmUgKHRoZSBhbnRpLWdhbWVzbG9wIGd1YXJkKQoiIiIKCmZyb20gLnBlcnNvbmFzIGltcG9ydCBQZXJzb25hLCBTVEFSVEVSX1BFUlNPTkFTLCBzeXN0ZW1fcHJvbXB0X2Zvcgpmcm9tIC5lbmdpbmUgaW1wb3J0IE5QQ0VuZ2luZSwgTlBDUmVzcG9uc2UKCl9fdmVyc2lvbl9fID0gIjAuMy4wIgoKX19hbGxfXyA9IFsKICAgICJQZXJzb25hIiwgIlNUQVJURVJfUEVSU09OQVMiLCAic3lzdGVtX3Byb21wdF9mb3IiLAogICAgIk5QQ0VuZ2luZSIsICJOUENSZXNwb25zZSIsCl0K',
    'npc_engine/personas.py': 'IiIiUGVyc29uYSBjYXJkcyBmb3IgdGhlIE5QQyBFbmdpbmUuCgpBIHBlcnNvbmEgY2FyZCBpcyB0aGUgY29uZGl0aW9uaW5nIGNvbnRleHQgdGhlIGRpZmZ1c2lvbiBtb2RlbCBzZWVzIHdoZW4gaXQKZGVub2lzZXMgYW4gTlBDIGxpbmU6IHdobyB0aGUgY2hhcmFjdGVyIGlzLCBob3cgdGhleSB0YWxrLCB3aGF0IHRoZXkga25vdy4KVGhlIGNhcmQgc3RheXMgQ0xFQU4gKG5ldmVyIG1hc2tlZCkg4oCUIG9ubHkgdGhlIE5QQydzIGxpbmUgaXMgZGVub2lzZWQuCiIiIgoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZAoKClJVTEVTID0gKAogICAgIlJ1bGVzOiBzdGF5IGluIGNoYXJhY3RlciBhdCBhbGwgdGltZXMuIFJlcGx5IHdpdGggMS0yIHNob3J0IHNwb2tlbiAiCiAgICAibGluZXMgb25seSAodW5kZXIgNDAgd29yZHMpLiBOZXZlciBicmVhayBjaGFyYWN0ZXIsIG5ldmVyIG1lbnRpb24gIgogICAgImJlaW5nIGFuIEFJLiIKKQoKCmRlZiBzeXN0ZW1fcHJvbXB0X2ZvcihuYW1lOiBzdHIsIGNhcmQ6IHN0ciwgd29ybGRfc3RhdGU6IHN0ciwKICAgICAgICAgICAgICAgICAgICAgIGhpc3Rvcnk6IGxpc3QgfCBOb25lID0gTm9uZSkgLT4gc3RyOgogICAgIiIiVEhFIGNhbm9uaWNhbCBOUEMgc3lzdGVtIHByb21wdCDigJQgdXNlZCBieSB0aGUgZW5naW5lIGF0IGluZmVyZW5jZSBBTkQKICAgIGJ5IHRoZSBmaW5lLXR1bmUgZGF0YSBidWlsZGVyLCBzbyB0cmFpbiBhbmQgaW5mZXJlbmNlIGZvcm1hdHMgbWF0Y2guIiIiCiAgICBwYXJ0cyA9IFtmIllvdSBhcmUge25hbWV9LiB7Y2FyZH0iXQogICAgaWYgaGlzdG9yeToKICAgICAgICBwYXJ0cy5hcHBlbmQoIlJlY2VudCBjb252ZXJzYXRpb246XG4iICsgIlxuIi5qb2luKGhpc3RvcnkpKQogICAgcGFydHMuYXBwZW5kKGYiQ3VycmVudCBzaXR1YXRpb246IHt3b3JsZF9zdGF0ZX0iKQogICAgcGFydHMuYXBwZW5kKFJVTEVTKQogICAgcmV0dXJuICJcblxuIi5qb2luKHBhcnRzKQoKCkBkYXRhY2xhc3MKY2xhc3MgUGVyc29uYToKICAgIG5hbWU6IHN0cgogICAgcm9sZTogc3RyCiAgICBwZXJzb25hbGl0eTogc3RyCiAgICBzcGVlY2hfc3R5bGU6IHN0cgogICAga25vd2xlZGdlOiBzdHIKICAgICMgU2FmZSBjYW5uZWQgbGluZXMgdXNlZCB3aGVuIG1vZGVsIGNvbmZpZGVuY2UgaXMgYmVsb3cgdGhyZXNob2xkLgogICAgZmFsbGJhY2tfbGluZXM6IGxpc3QgPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKCiAgICBkZWYgY2FyZF90ZXh0KHNlbGYpIC0+IHN0cjoKICAgICAgICByZXR1cm4gKAogICAgICAgICAgICBmIntzZWxmLnJvbGV9LiBQZXJzb25hbGl0eToge3NlbGYucGVyc29uYWxpdHl9LiAiCiAgICAgICAgICAgIGYiU3BlZWNoIHN0eWxlOiB7c2VsZi5zcGVlY2hfc3R5bGV9LiAiCiAgICAgICAgICAgIGYiV2hhdCB5b3Uga25vdzoge3NlbGYua25vd2xlZGdlfS4iCiAgICAgICAgKQoKICAgIGRlZiBzeXN0ZW1fcHJvbXB0KHNlbGYsIHdvcmxkX3N0YXRlOiBzdHIsIGhpc3Rvcnk6IGxpc3QgfCBOb25lID0gTm9uZSkgLT4gc3RyOgogICAgICAgIHJldHVybiBzeXN0ZW1fcHJvbXB0X2ZvcihzZWxmLm5hbWUsIHNlbGYuY2FyZF90ZXh0KCksIHdvcmxkX3N0YXRlLCBoaXN0b3J5KQoKClNUQVJURVJfUEVSU09OQVMgPSBbCiAgICBQZXJzb25hKAogICAgICAgIG5hbWU9IkJvcmluIElyb25maXN0IiwKICAgICAgICByb2xlPSJ0aGUgd2VhcnkgYmxhY2tzbWl0aCBvZiBNaWxsYnJvb2sgdmlsbGFnZSIsCiAgICAgICAgcGVyc29uYWxpdHk9ImdydWZmIGJ1dCBraW5kLCB0aXJlZCwgZHJ5IGh1bW9yLCBwcm90ZWN0aXZlIG9mIHRoZSB2aWxsYWdlIiwKICAgICAgICBzcGVlY2hfc3R5bGU9InNob3J0IGNsaXBwZWQgc2VudGVuY2VzLCBjYWxscyBwZW9wbGUgJ2xhZCcgb3IgJ2xhc3MnLCBvY2Nhc2lvbmFsIGZvcmdlIG1ldGFwaG9ycyIsCiAgICAgICAga25vd2xlZGdlPSJ3ZWFwb25zLCBhcm1vciwgdGhlIG1pbGxlcidzIGJyb2tlbiBjYXJ0LCBydW1vcnMgb2YgYmFuZGl0cyBvbiB0aGUgbm9ydGggcm9hZCIsCiAgICAgICAgZmFsbGJhY2tfbGluZXM9WyJBeWUuIiwgIkhtbS4gQ2FuJ3Qgc2F5LiIsICJTdGVlbCdzIGhvbmVzdC4gUGVvcGxlIGxlc3Mgc28uIl0sCiAgICApLAogICAgUGVyc29uYSgKICAgICAgICBuYW1lPSJNaXJhIFRoaXN0bGVkb3duIiwKICAgICAgICByb2xlPSJ0aGUgY2hlZXJmdWwgaW5ua2VlcGVyIG9mIHRoZSBSdXN0eSBLZXR0bGUiLAogICAgICAgIHBlcnNvbmFsaXR5PSJ3YXJtLCBjaGF0dHksIGdvc3NpcHksIG1vdGhlcmx5IHRvIHRyYXZlbGVycyIsCiAgICAgICAgc3BlZWNoX3N0eWxlPSJsb25nZXIgZmxvd2luZyBzZW50ZW5jZXMsIGV4Y2xhbWF0aW9ucywgYXNrcyBxdWVzdGlvbnMgYmFjaywgZm9vZCBtZXRhcGhvcnMiLAogICAgICAgIGtub3dsZWRnZT0idmlsbGFnZSBnb3NzaXAsIHJvb20gcmF0ZXMgKDUgY29wcGVyIGEgbmlnaHQpLCB0aGUgc3RldyByZWNpcGUsIHdobyBvd2VzIGhlciBtb25leSIsCiAgICAgICAgZmFsbGJhY2tfbGluZXM9WyJPaCwgZGVhcmllISIsICJOb3cgdGhhdCdzIGEgdGFsZS4iLCAiU2l0LCBzaXQhIFN0ZXcncyBvbi4iXSwKICAgICksCiAgICBQZXJzb25hKAogICAgICAgIG5hbWU9IkNhcHRhaW4gU2VyYSBWb3NzIiwKICAgICAgICByb2xlPSJjYXB0YWluIG9mIHRoZSB2aWxsYWdlIGd1YXJkIiwKICAgICAgICBwZXJzb25hbGl0eT0iZGlzY2lwbGluZWQsIHN1c3BpY2lvdXMgb2Ygc3RyYW5nZXJzLCBzZWNyZXRseSB3b3JyaWVkIGFib3V0IHRoZSBiYW5kaXRzIiwKICAgICAgICBzcGVlY2hfc3R5bGU9InRlcnNlLCBkaXJlY3QsIG1pbGl0YXJ5IHBocmFzaW5nLCBubyB3YXN0ZWQgd29yZHMiLAogICAgICAgIGtub3dsZWRnZT0iZ3VhcmQgcm90YXRpb25zLCB0aGUgbm9ydGggcm9hZCBhbWJ1c2ggbGFzdCB3ZWVrLCB0aGUgbWF5b3IncyBvcmRlcnMiLAogICAgICAgIGZhbGxiYWNrX2xpbmVzPVsiU3RhdGUgeW91ciBidXNpbmVzcy4iLCAiTm90ZWQuIiwgIk1vdmUgYWxvbmcuIl0sCiAgICApLAogICAgUGVyc29uYSgKICAgICAgICBuYW1lPSJGZWxpeCBRdWlja2ZpbmdlcnMiLAogICAgICAgIHJvbGU9ImEgc2hhZHkgdHJhdmVsaW5nIG1lcmNoYW50IiwKICAgICAgICBwZXJzb25hbGl0eT0ib2lseSBjaGFybSwgYWx3YXlzIHNlbGxpbmcsIG5lcnZvdXMgd2hlbiBhc2tlZCBhYm91dCBoaXMgc3VwcGxpZXJzIiwKICAgICAgICBzcGVlY2hfc3R5bGU9ImZhc3QgcGF0dGVyLCBzdXBlcmxhdGl2ZXMsICdteSBmcmllbmQnLCBkZWZsZWN0cyBxdWVzdGlvbnMgd2l0aCBvZmZlcnMiLAogICAgICAgIGtub3dsZWRnZT0iaGlzIHdhcmVzIChwb3Rpb25zLCBtYXBzLCAnZ2VudWluZScgcmVsaWNzKSwgdGhlIHJvYWRzIGJldHdlZW4gdG93bnMsIHByaWNlcyIsCiAgICAgICAgZmFsbGJhY2tfbGluZXM9WyJNeSBmcmllbmQhIiwgIkZvciB5b3UsIGEgc3BlY2lhbCBwcmljZS4iLCAiQWgsIGJ1c2luZXNzIGlzIGJ1c2luZXNzLiJdLAogICAgKSwKICAgIFBlcnNvbmEoCiAgICAgICAgbmFtZT0iRWxkZXIgV3JlbiIsCiAgICAgICAgcm9sZT0idGhlIHZpbGxhZ2UgaGVhbGVyIGFuZCBoZXJiYWxpc3QiLAogICAgICAgIHBlcnNvbmFsaXR5PSJjYWxtLCB3aXNlLCBzcGVha3MgaW4gcGxhbnQgbWV0YXBob3JzLCBnZW50bGUgYnV0IGZpcm0iLAogICAgICAgIHNwZWVjaF9zdHlsZT0ic2xvdyBtZWFzdXJlZCBzcGVlY2gsIHJoZXRvcmljYWwgcXVlc3Rpb25zLCByZWZlcmVuY2VzIHRvIGhlcmJzIGFuZCBzZWFzb25zIiwKICAgICAgICBrbm93bGVkZ2U9ImhlYWxpbmcgaGVyYnMsIHRoZSBmZXZlciBnb2luZyBhcm91bmQsIHBvdWx0aWNlcywgb2xkIHZpbGxhZ2UgaGlzdG9yeSIsCiAgICAgICAgZmFsbGJhY2tfbGluZXM9WyJUaGUgYm9keSBrbm93cy4iLCAiUmVzdCwgY2hpbGQuIiwgIk5hdHVyZSBwcm92aWRlcy4iXSwKICAgICksCiAgICBQZXJzb25hKAogICAgICAgIG5hbWU9IlBpcCBVbmRlcmJvdWdoIiwKICAgICAgICByb2xlPSJhIG5lcnZvdXMgeW91bmcgbWVzc2VuZ2VyIiwKICAgICAgICBwZXJzb25hbGl0eT0iYW54aW91cywgZWFnZXIgdG8gcGxlYXNlLCBvdXQgb2YgYnJlYXRoLCB0YWxrcyB0b28gZmFzdCB3aGVuIHNjYXJlZCIsCiAgICAgICAgc3BlZWNoX3N0eWxlPSJmcmFnbWVudHMsIHN0YW1tZXJpbmcgd2hlbiBuZXJ2b3VzLCByZXBlYXRzIGhpbXNlbGYiLAogICAgICAgIGtub3dsZWRnZT0idGhlIHJvYWRzLCB3aG8gc2VudCBoaW0sIHRoZSBzZWFsZWQgbGV0dGVyIGhlIGNhcnJpZXMgKGhlIGhhc24ndCByZWFkIGl0KSIsCiAgICAgICAgZmFsbGJhY2tfbGluZXM9WyJJ4oCUIEkganVzdCBkZWxpdmVyISIsICJQbGVhc2UgZG9uJ3Qgc2hvb3QgdGhlIG1lc3NlbmdlciEiLCAiR290dGEgcnVuISJdLAogICAgKSwKXQo=',
    'npc_engine/engine.py': 'IiIiTlBDRW5naW5lIOKAlCBwYXJhbGxlbCBjYW5kaWRhdGUgZ2VuZXJhdGlvbiArIGNvbmZpZGVuY2UgcmFua2luZyBmb3IgTlBDIGxpbmVzLgoKSG93IGl0IHdvcmtzIChkaWZmdXNpb24td2lzZSk6CjEuIFRoZSBwZXJzb25hIGNhcmQgKyB3b3JsZCBzdGF0ZSArIHBsYXllciBsaW5lIGZvcm0gdGhlIHByb21wdC4gVGhlIHByb21wdCBpcwogICBrZXB0IGZ1bGx5IGNsZWFuIChuZXZlciBtYXNrZWQpLgoyLiBXZSBhcHBlbmQgYSBjYW52YXMgb2YgW01BU0tdIHRva2VucyBhbmQgZXhwYW5kIGl0IHRvIGEgYmF0Y2ggb2YgTiBjYW52YXNlcy4KICAgQWxsIE4gY2FuZGlkYXRlcyBhcmUgZGVub2lzZWQgaW4gT05FIGJhdGNoZWQgZGlmZnVzaW9uIHBhc3Mg4oCUIEFSIHdvdWxkIG5lZWQKICAgTiBzZXF1ZW50aWFsIGdlbmVyYXRpb25zLiBEaXZlcnNpdHkgY29tZXMgZnJvbSB0ZW1wZXJhdHVyZS90b3AtcCBzYW1wbGluZy4KMy4gQSBzaW5nbGUgc2NvcmluZyBmb3J3YXJkIHBhc3MgY29tcHV0ZXMsIGZvciBlYWNoIGNhbmRpZGF0ZSwgdGhlIG1lYW4KICAgbWF4LXByb2JhYmlsaXR5IG92ZXIgaXRzIGdlbmVyYXRlZCB0b2tlbnMgKGNvbmZpZGVuY2UpIHBsdXMgdGhlIHdlYWtlc3QKICAgdG9rZW4ncyBwcm9iYWJpbGl0eS4gQ2FuZGlkYXRlcyBhcmUgcmFua2VkIGJ5IGNvbmZpZGVuY2UuCjQuIElmIHRoZSBiZXN0IGNhbmRpZGF0ZSdzIGNvbmZpZGVuY2UgaXMgYmVsb3cgdGhlIHRocmVzaG9sZCwgdGhlIGVuZ2luZQogICByZXR1cm5zIG9uZSBvZiB0aGUgcGVyc29uYSdzIGNhbm5lZCBmYWxsYmFjayBsaW5lcyBpbnN0ZWFkIChmbGFnZ2VkKSwgc28gYQogICBsb3ctY29uZmlkZW5jZSBnZW5lcmF0aW9uIG5ldmVyIHJlYWNoZXMgdGhlIHBsYXllci4KIiIiCgppbXBvcnQgdGltZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgoKZnJvbSAucGVyc29uYXMgaW1wb3J0IFBlcnNvbmEKCkRFRkFVTFRfTU9ERUwgPSAiRHJlYW0tb3JnL0RyZWFtLXYwLUluc3RydWN0LTdCIgpNQVNLX1RPS0VOID0gIjx8bWFza3w+IgoKCkBkYXRhY2xhc3MKY2xhc3MgTlBDUmVzcG9uc2U6CiAgICB0ZXh0OiBzdHIgICAgICAgICAgICAgICAgICMgdGhlIGxpbmUgdGhlIGdhbWUgc2hvdWxkIHNoaXAKICAgIGNvbmZpZGVuY2U6IGZsb2F0ICAgICAgICAgIyBjb25maWRlbmNlIG9mIHRoZSBzaGlwcGVkIGxpbmUgKDEuMCBmb3IgZmFsbGJhY2spCiAgICB1c2VkX2ZhbGxiYWNrOiBib29sICAgICAgICMgVHJ1ZSBpZiB0aGUgbW9kZWwgd2FzIHRvbyB1bnN1cmUgYW5kIHdlIGNhbm5lZCBpdAogICAgY2FuZGlkYXRlczogbGlzdCA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KSAgIyBbKHRleHQsIGNvbmZpZGVuY2UpLCAuLi5dIHJhbmtlZAogICAgbGF0ZW5jeV9tczogZmxvYXQgPSAwLjAgICAjIHdhbGwgdGltZSBmb3IgdGhlIHdob2xlIHJlc3BvbmQoKSBjYWxsCgoKY2xhc3MgTlBDRW5naW5lOgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgbW9kZWxfaWQ6IHN0ciA9IERFRkFVTFRfTU9ERUwsCiAgICAgICAgZGV2aWNlOiBzdHIgfCBOb25lID0gTm9uZSwKICAgICAgICBsb2FkX2luXzhiaXQ6IGJvb2wgPSBGYWxzZSwKICAgICAgICB0b3JjaF9kdHlwZTogdG9yY2guZHR5cGUgfCBOb25lID0gTm9uZSwKICAgICAgICBhZGFwdGVyX3BhdGg6IHN0ciB8IE5vbmUgPSBOb25lLAogICAgKToKICAgICAgICAiIiJMb2FkIHRoZSBkaWZmdXNpb24gTE0uIE9uIGEgMTZHQiBHUFUgdXNlIGxvYWRfaW5fOGJpdD1UcnVlOyBvbiAyNEdCKwogICAgICAgIGZwMTYvYmYxNiBpcyBmaW5lLiBhZGFwdGVyX3BhdGggbG9hZHMgYSBMb1JBIGFkYXB0ZXIgb24gdG9wLiIiIgogICAgICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvTW9kZWwsIEF1dG9Ub2tlbml6ZXIKCiAgICAgICAgc2VsZi5kZXZpY2UgPSBkZXZpY2Ugb3IgKCJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIpCiAgICAgICAga3dhcmdzID0geyJ0cnVzdF9yZW1vdGVfY29kZSI6IFRydWV9CiAgICAgICAgaWYgbG9hZF9pbl84Yml0OgogICAgICAgICAgICAjIE1vZGVybiB0cmFuc2Zvcm1lcnM6IDgtYml0IGxvYWRpbmcgZ29lcyB0aHJvdWdoIGEgcXVhbnRpemF0aW9uCiAgICAgICAgICAgICMgY29uZmlnICsgZGV2aWNlX21hcC4gUGFzc2luZyBsb2FkX2luXzhiaXQ9VHJ1ZSBkaXJlY3RseSBsZWFrcyB0aGUKICAgICAgICAgICAgIyBrd2FyZyBpbnRvIHRoZSBjdXN0b20gbW9kZWwncyBfX2luaXRfXyBhbmQgY3Jhc2hlcyBpdC4KICAgICAgICAgICAgZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IEJpdHNBbmRCeXRlc0NvbmZpZwogICAgICAgICAgICBrd2FyZ3NbInF1YW50aXphdGlvbl9jb25maWciXSA9IEJpdHNBbmRCeXRlc0NvbmZpZyhsb2FkX2luXzhiaXQ9VHJ1ZSkKICAgICAgICAgICAga3dhcmdzWyJkZXZpY2VfbWFwIl0gPSAiYXV0byIKICAgICAgICBlbHNlOgogICAgICAgICAgICBrd2FyZ3NbInRvcmNoX2R0eXBlIl0gPSB0b3JjaF9kdHlwZSBvciB0b3JjaC5mbG9hdDE2CiAgICAgICAgc2VsZi5tb2RlbCA9IEF1dG9Nb2RlbC5mcm9tX3ByZXRyYWluZWQobW9kZWxfaWQsICoqa3dhcmdzKQogICAgICAgIGlmIG5vdCBsb2FkX2luXzhiaXQ6CiAgICAgICAgICAgIHNlbGYubW9kZWwgPSBzZWxmLm1vZGVsLnRvKHNlbGYuZGV2aWNlKQogICAgICAgIGlmIGFkYXB0ZXJfcGF0aDoKICAgICAgICAgICAgZnJvbSBwZWZ0IGltcG9ydCBQZWZ0TW9kZWwKICAgICAgICAgICAgc2VsZi5tb2RlbCA9IFBlZnRNb2RlbC5mcm9tX3ByZXRyYWluZWQoc2VsZi5tb2RlbCwgYWRhcHRlcl9wYXRoKQogICAgICAgIHNlbGYubW9kZWwuZXZhbCgpCiAgICAgICAgc2VsZi50b2tlbml6ZXIgPSBBdXRvVG9rZW5pemVyLmZyb21fcHJldHJhaW5lZChtb2RlbF9pZCwgdHJ1c3RfcmVtb3RlX2NvZGU9VHJ1ZSkKICAgICAgICBzZWxmLm1hc2tfdG9rZW5faWQgPSBzZWxmLnRva2VuaXplci5jb252ZXJ0X3Rva2Vuc190b19pZHMoTUFTS19UT0tFTikKICAgICAgICBlb3MgPSBzZWxmLnRva2VuaXplci5lb3NfdG9rZW5faWQKICAgICAgICBzZWxmLmVvc19pZHMgPSBzZXQoZW9zIGlmIGlzaW5zdGFuY2UoZW9zLCBsaXN0KSBlbHNlIFtlb3NdKQoKICAgICMgLS0tLSBwcm9tcHQgYnVpbGRpbmcgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgZGVmIF9wcm9tcHRfaWRzKHNlbGYsIHBlcnNvbmE6IFBlcnNvbmEsIHdvcmxkX3N0YXRlOiBzdHIsIHBsYXllcl9saW5lOiBzdHIsCiAgICAgICAgICAgICAgICAgICAgaGlzdG9yeTogbGlzdCB8IE5vbmUgPSBOb25lKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgbWVzc2FnZXMgPSBbCiAgICAgICAgICAgIHsicm9sZSI6ICJzeXN0ZW0iLCAiY29udGVudCI6IHBlcnNvbmEuc3lzdGVtX3Byb21wdCh3b3JsZF9zdGF0ZSwgaGlzdG9yeSl9LAogICAgICAgICAgICB7InJvbGUiOiAidXNlciIsICJjb250ZW50IjogcGxheWVyX2xpbmV9LAogICAgICAgIF0KICAgICAgICBpZHMgPSBzZWxmLnRva2VuaXplci5hcHBseV9jaGF0X3RlbXBsYXRlKAogICAgICAgICAgICBtZXNzYWdlcywgdG9rZW5pemU9VHJ1ZSwgYWRkX2dlbmVyYXRpb25fcHJvbXB0PVRydWUKICAgICAgICApCiAgICAgICAgcmV0dXJuIHRvcmNoLnRlbnNvcihbaWRzXSwgZHR5cGU9dG9yY2gubG9uZywgZGV2aWNlPXNlbGYuZGV2aWNlKQoKICAgICMgLS0tLSBjb25maWRlbmNlIHNjb3JpbmcgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgQHRvcmNoLm5vX2dyYWQoKQogICAgZGVmIF9zY29yZV9jYW5kaWRhdGVzKHNlbGYsIHNlcXM6IHRvcmNoLlRlbnNvciwgcHJvbXB0X2xlbjogaW50KToKICAgICAgICAiIiJPbmUgZm9yd2FyZCBwYXNzOyBwZXIgY2FuZGlkYXRlLCAobWVhbiBtYXgtcHJvYiwgbWluIG1heC1wcm9iKSBvdmVyCiAgICAgICAgdGhlIGdlbmVyYXRlZCBzcGFuICh1cCB0byBpdHMgZmlyc3QgRU9TKS4iIiIKICAgICAgICBsb2dpdHMgPSBzZWxmLm1vZGVsKHNlcXMpLmxvZ2l0cy5mbG9hdCgpCiAgICAgICAgcHJvYnMgPSBGLnNvZnRtYXgobG9naXRzLCBkaW09LTEpCiAgICAgICAgbWF4cCwgXyA9IHByb2JzLm1heChkaW09LTEpICAjIChCLCBMKQogICAgICAgIG91dCA9IFtdCiAgICAgICAgZm9yIGIgaW4gcmFuZ2Uoc2Vxcy5zaGFwZVswXSk6CiAgICAgICAgICAgIHNwYW4gPSBbXQogICAgICAgICAgICBmb3IgdCBpbiByYW5nZShwcm9tcHRfbGVuLCBzZXFzLnNoYXBlWzFdKToKICAgICAgICAgICAgICAgIHRvayA9IGludChzZXFzW2IsIHRdKQogICAgICAgICAgICAgICAgaWYgdG9rIGluIHNlbGYuZW9zX2lkcyBvciB0b2sgPT0gc2VsZi5tYXNrX3Rva2VuX2lkOgogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICBzcGFuLmFwcGVuZChmbG9hdChtYXhwW2IsIHRdKSkKICAgICAgICAgICAgaWYgc3BhbjoKICAgICAgICAgICAgICAgIG91dC5hcHBlbmQoKHN1bShzcGFuKSAvIGxlbihzcGFuKSwgbWluKHNwYW4pKSkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIG91dC5hcHBlbmQoKDAuMCwgMC4wKSkKICAgICAgICByZXR1cm4gb3V0CgogICAgIyAtLS0tIG1haW4gZW50cnkgcG9pbnQgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIEB0b3JjaC5ub19ncmFkKCkKICAgIGRlZiByZXNwb25kKAogICAgICAgIHNlbGYsCiAgICAgICAgcGVyc29uYTogUGVyc29uYSwKICAgICAgICB3b3JsZF9zdGF0ZTogc3RyLAogICAgICAgIHBsYXllcl9saW5lOiBzdHIsCiAgICAgICAgaGlzdG9yeTogbGlzdCB8IE5vbmUgPSBOb25lLAogICAgICAgIG5fY2FuZGlkYXRlczogaW50ID0gOCwKICAgICAgICBzdGVwczogaW50ID0gMzIsCiAgICAgICAgbWF4X25ld190b2tlbnM6IGludCA9IDY0LAogICAgICAgIHRlbXBlcmF0dXJlOiBmbG9hdCA9IDAuOCwKICAgICAgICB0b3BfcDogZmxvYXQgPSAwLjk1LAogICAgICAgIGNvbmZpZGVuY2VfdGhyZXNob2xkOiBmbG9hdCA9IDAuNDUsCiAgICApIC0+IE5QQ1Jlc3BvbnNlOgogICAgICAgIHQwID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIHByb21wdF9pZHMgPSBzZWxmLl9wcm9tcHRfaWRzKHBlcnNvbmEsIHdvcmxkX3N0YXRlLCBwbGF5ZXJfbGluZSwgaGlzdG9yeSkKICAgICAgICBwcm9tcHRfbGVuID0gcHJvbXB0X2lkcy5zaGFwZVsxXQoKICAgICAgICBjYW52YXMgPSB0b3JjaC5mdWxsKAogICAgICAgICAgICAoMSwgbWF4X25ld190b2tlbnMpLCBzZWxmLm1hc2tfdG9rZW5faWQsIGR0eXBlPXRvcmNoLmxvbmcsIGRldmljZT1zZWxmLmRldmljZQogICAgICAgICkKICAgICAgICBpbnB1dF9pZHMgPSB0b3JjaC5jYXQoW3Byb21wdF9pZHMsIGNhbnZhc10sIGRpbT0xKQogICAgICAgICMgcGFyYWxsZWwgY2FuZGlkYXRlczogTiBjYW52YXNlcywgT05FIGJhdGNoZWQgZGVub2lzaW5nIHJ1bgogICAgICAgIGlucHV0X2lkcyA9IGlucHV0X2lkcy5yZXBlYXQobl9jYW5kaWRhdGVzLCAxKQogICAgICAgIGF0dG4gPSB0b3JjaC5vbmVzX2xpa2UoaW5wdXRfaWRzKQoKICAgICAgICBnZW4gPSBzZWxmLm1vZGVsLmRpZmZ1c2lvbl9nZW5lcmF0ZSBpZiBoYXNhdHRyKHNlbGYubW9kZWwsICJkaWZmdXNpb25fZ2VuZXJhdGUiKSBlbHNlIHNlbGYubW9kZWwuZ2VuZXJhdGUKICAgICAgICBvdXQgPSBnZW4oCiAgICAgICAgICAgIGlucHV0X2lkcywKICAgICAgICAgICAgYXR0ZW50aW9uX21hc2s9YXR0biwKICAgICAgICAgICAgbWF4X25ld190b2tlbnM9bWF4X25ld190b2tlbnMsCiAgICAgICAgICAgIHN0ZXBzPXN0ZXBzLAogICAgICAgICAgICB0ZW1wZXJhdHVyZT10ZW1wZXJhdHVyZSwKICAgICAgICAgICAgdG9wX3A9dG9wX3AsCiAgICAgICAgICAgIHJldHVybl9kaWN0X2luX2dlbmVyYXRlPVRydWUsCiAgICAgICAgKQogICAgICAgIHNlcXMgPSBvdXQuc2VxdWVuY2VzIGlmIGhhc2F0dHIob3V0LCAic2VxdWVuY2VzIikgZWxzZSBvdXQKCiAgICAgICAgc2NvcmVzID0gc2VsZi5fc2NvcmVfY2FuZGlkYXRlcyhzZXFzLCBwcm9tcHRfbGVuKQogICAgICAgIHJhbmtlZCA9IFtdCiAgICAgICAgZm9yIGIgaW4gcmFuZ2Uoc2Vxcy5zaGFwZVswXSk6CiAgICAgICAgICAgIHRleHQgPSBzZWxmLl9kZWNvZGVfbGluZShzZXFzW2JdLCBwcm9tcHRfbGVuKQogICAgICAgICAgICBpZiB0ZXh0OgogICAgICAgICAgICAgICAgcmFua2VkLmFwcGVuZCgodGV4dCwgc2NvcmVzW2JdWzBdLCBzY29yZXNbYl1bMV0pKQogICAgICAgICMgZGVkdXBlIGlkZW50aWNhbCBsaW5lcywgcmFuayBieSBtZWFuIGNvbmZpZGVuY2UgdGhlbiB3ZWFrZXN0IGxpbmsKICAgICAgICBzZWVuLCB1bmlxID0gc2V0KCksIFtdCiAgICAgICAgZm9yIHRleHQsIG1lYW5fYywgbWluX2MgaW4gc29ydGVkKHJhbmtlZCwga2V5PWxhbWJkYSByOiAoclsxXSwgclsyXSksIHJldmVyc2U9VHJ1ZSk6CiAgICAgICAgICAgIGlmIHRleHQgbm90IGluIHNlZW46CiAgICAgICAgICAgICAgICBzZWVuLmFkZCh0ZXh0KQogICAgICAgICAgICAgICAgdW5pcS5hcHBlbmQoKHRleHQsIG1lYW5fYykpCiAgICAgICAgbGF0ZW5jeV9tcyA9ICh0aW1lLnBlcmZfY291bnRlcigpIC0gdDApICogMTAwMAoKICAgICAgICBpZiBub3QgdW5pcSBvciB1bmlxWzBdWzFdIDwgY29uZmlkZW5jZV90aHJlc2hvbGQ6CiAgICAgICAgICAgIGltcG9ydCByYW5kb20KICAgICAgICAgICAgZmFsbGJhY2sgPSByYW5kb20uY2hvaWNlKHBlcnNvbmEuZmFsbGJhY2tfbGluZXMpIGlmIHBlcnNvbmEuZmFsbGJhY2tfbGluZXMgZWxzZSAiLi4uIgogICAgICAgICAgICByZXR1cm4gTlBDUmVzcG9uc2UoCiAgICAgICAgICAgICAgICB0ZXh0PWZhbGxiYWNrLCBjb25maWRlbmNlPTEuMCwgdXNlZF9mYWxsYmFjaz1UcnVlLAogICAgICAgICAgICAgICAgY2FuZGlkYXRlcz11bmlxLCBsYXRlbmN5X21zPWxhdGVuY3lfbXMsCiAgICAgICAgICAgICkKICAgICAgICByZXR1cm4gTlBDUmVzcG9uc2UoCiAgICAgICAgICAgIHRleHQ9dW5pcVswXVswXSwgY29uZmlkZW5jZT11bmlxWzBdWzFdLCB1c2VkX2ZhbGxiYWNrPUZhbHNlLAogICAgICAgICAgICBjYW5kaWRhdGVzPXVuaXEsIGxhdGVuY3lfbXM9bGF0ZW5jeV9tcywKICAgICAgICApCgogICAgZGVmIF9kZWNvZGVfbGluZShzZWxmLCBzZXE6IHRvcmNoLlRlbnNvciwgcHJvbXB0X2xlbjogaW50KSAtPiBzdHI6CiAgICAgICAgaWRzID0gW10KICAgICAgICBmb3IgdCBpbiByYW5nZShwcm9tcHRfbGVuLCBzZXEuc2hhcGVbMF0pOgogICAgICAgICAgICB0b2sgPSBpbnQoc2VxW3RdKQogICAgICAgICAgICBpZiB0b2sgaW4gc2VsZi5lb3NfaWRzIG9yIHRvayA9PSBzZWxmLm1hc2tfdG9rZW5faWQ6CiAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICBpZHMuYXBwZW5kKHRvaykKICAgICAgICByZXR1cm4gc2VsZi50b2tlbml6ZXIuZGVjb2RlKGlkcywgc2tpcF9zcGVjaWFsX3Rva2Vucz1UcnVlKS5zdHJpcCgpCg==',
    'npc_engine/games/__init__.py': '',
    'npc_engine/games/base_of_three.py': 'IiIiQmFzZSBvZiBUaHJlZSDigJQgYSB0aW55IHBsYXlhYmxlIHN1cnZpdm9yLWNhbXAgZ2FtZSBvbiB0aGUgTlBDIGVuZ2luZS4KClRocmVlIHN1cnZpdm9ycywgdGhyZWUgZGF5cy4gRWFjaCBtb3JuaW5nIHlvdSBhc3NpZ24gdGFza3MgKHNjYXZlbmdlIC8gZ3VhcmQKLyByZXN0KTsgZXZlbnRzIGxhbmQ7IGVhY2ggZXZlbmluZyBldmVyeSBzdXJ2aXZvciBzYXlzIE9ORSBsaW5lIGdlbmVyYXRlZApmcm9tIHRoZWlyIGxpdmUgc3RhdGUgY2FyZCAodHJhaXRzLCBtb3JhbGUsIGluanVyaWVzLCBpbmZlY3Rpb24pLiBUaGUgR0FNRQpkZWNpZGVzIGV2ZXJ5IGZhY3Qg4oCUIGZvb2QsIHdvdW5kcywgd2hvIGxpdmVzIOKAlCB0aGUgZW5naW5lIG9ubHkgd29yZHMgaXQuCgpXaW46IHJlYWNoIGRheSA0IHdpdGggZXZlcnlvbmUgYWxpdmUgYW5kIHN0aWxsIGluIGNhbXAuCkxvc2U6IGEgc3Vydml2b3IgZGllcyBvZiBhbiB1bnRyZWF0ZWQgaW5mZWN0aW9uLCBvciBzb21lb25lJ3MgbW9yYWxlIGhpdHMKemVybyBhbmQgdGhleSB3YWxrIG91dCBpbiB0aGUgbmlnaHQuCgpUaGUgZ2FtZSBsb2dpYyBpcyBwdXJlIFB5dGhvbiAoaW5qZWN0YWJsZSBzcGVha2VyICsgUk5HKSBzbyBpdCBpcyBmdWxseQp0ZXN0YWJsZSB3aXRob3V0IGEgbW9kZWwuIEluIENvbGFiLCB0aGUgc3BlYWtlciB3cmFwcyBOUENFbmdpbmUucmVzcG9uZCgpLgoiIiIKCmltcG9ydCByYW5kb20KCmZyb20gLi5wZXJzb25hcyBpbXBvcnQgUGVyc29uYQoKQkFSS19QUk9NUFQgPSAoCiAgICAiVGhlIGdyb3VwIGlzIGdhdGhlcmVkIGF0IGNhbXAgZm9yIHRoZSBldmVuaW5nLiBTYXkgb25lIG9yIHR3byBzaG9ydCAiCiAgICAibGluZXMsIGluIGNoYXJhY3RlciwgYWJvdXQgaG93IHlvdSBhcmUgZG9pbmcgdG9uaWdodC4iCikKClRBU0tTID0gKCJzY2F2ZW5nZSIsICJndWFyZCIsICJyZXN0IikKCgpjbGFzcyBTdXJ2aXZvcjoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBwZXJzb25hOiBQZXJzb25hKToKICAgICAgICBzZWxmLnBlcnNvbmEgPSBwZXJzb25hCiAgICAgICAgc2VsZi5tb3JhbGUgPSA2ICAgICAgICAgICMgMCA9IHdhbGtzIG91dCwgMTAgPSB0aHJpdmluZwogICAgICAgIHNlbGYuaGVhbHRoID0gIm9rIiAgICAgICAjIG9rIHwgaHVydCB8IGluZmVjdGVkCiAgICAgICAgc2VsZi50b2RheSA9ICJjYW1wIGR1dHkiICMgd2hhdCBoYXBwZW5lZCB0byB0aGVtIHRvZGF5IChmb3IgdGhlIGNhcmQpCiAgICAgICAgc2VsZi50YXNrID0gTm9uZQogICAgICAgIHNlbGYuYWxpdmUgPSBUcnVlCiAgICAgICAgc2VsZi5wcmVzZW50ID0gVHJ1ZQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIG5hbWUoc2VsZik6CiAgICAgICAgcmV0dXJuIHNlbGYucGVyc29uYS5uYW1lCgogICAgZGVmIGNvbmRpdGlvbl90ZXh0KHNlbGYpOgogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJvayI6ICJ1bmluanVyZWQiLAogICAgICAgICAgICAiaHVydCI6ICJodXJ0IChzY3JhdGNoZWQgdXAgYW5kIHNvcmUpIiwKICAgICAgICAgICAgImluZmVjdGVkIjogImluZmVjdGVkIOKAlCBmZXZlcmlzaCwgYml0dGVuIHR3byBkYXlzIGFnbywgZ2V0dGluZyB3b3JzZSIsCiAgICAgICAgfVtzZWxmLmhlYWx0aF0KCiAgICBkZWYgbW9yYWxlX3dvcmQoc2VsZik6CiAgICAgICAgaWYgc2VsZi5tb3JhbGUgPj0gODoKICAgICAgICAgICAgcmV0dXJuICJob3BlZnVsIgogICAgICAgIGlmIHNlbGYubW9yYWxlID49IDU6CiAgICAgICAgICAgIHJldHVybiAic3RlYWR5IgogICAgICAgIGlmIHNlbGYubW9yYWxlID49IDM6CiAgICAgICAgICAgIHJldHVybiAic2hha2VuIgogICAgICAgIHJldHVybiAiYXQgYnJlYWtpbmcgcG9pbnQiCgoKZGVmIHN0YXJ0ZXJfc3Vydml2b3JzKCk6CiAgICByZXR1cm4gWwogICAgICAgIFN1cnZpdm9yKFBlcnNvbmEoCiAgICAgICAgICAgIG5hbWU9Ikpvc3MgTWFyZWsiLCByb2xlPSJ0aGUgY2FtcCdzIG1lZGljLCBleC1hcm15IiwKICAgICAgICAgICAgcGVyc29uYWxpdHk9ImNhbG0sIGRyeS1odW1vcmVkLCBwdXRzIGV2ZXJ5b25lIGVsc2UgZmlyc3QiLAogICAgICAgICAgICBzcGVlY2hfc3R5bGU9InNob3J0LCBkcnksIHVuZGVyc3RhdGVkIiwKICAgICAgICAgICAga25vd2xlZGdlPSJmaXJzdCBhaWQsIGFuZCBleGFjdGx5IHdobyBpcyBodXJ0IGFuZCBob3cgYmFkbHkiLAogICAgICAgICAgICBmYWxsYmFja19saW5lcz1bIldlJ2xsIG1hbmFnZS4gV2UgYWx3YXlzIGRvLiIsICJPbmUgdGhpbmcgYXQgYSB0aW1lLCBwZW9wbGUuIl0sCiAgICAgICAgKSksCiAgICAgICAgU3Vydml2b3IoUGVyc29uYSgKICAgICAgICAgICAgbmFtZT0iUnV0aCBDYWxsYWhhbiIsIHJvbGU9InRoZSBjYW1wJ3MgaHVudGVyIGFuZCBzY2F2ZW5nZXIiLAogICAgICAgICAgICBwZXJzb25hbGl0eT0iYmx1bnQsIGJyYXZlLCBpbXBhdGllbnQgd2l0aCBwYW5pYyIsCiAgICAgICAgICAgIHNwZWVjaF9zdHlsZT0idGVyc2UgYW5kIHBsYWluc3Bva2VuIiwKICAgICAgICAgICAga25vd2xlZGdlPSJ0aGUgd29vZHMsIHRoZSByb2FkcywgYW5kIHdoZXJlIHN1cHBsaWVzIHN0aWxsIGFyZSIsCiAgICAgICAgICAgIGZhbGxiYWNrX2xpbmVzPVsiTGVzcyB0YWxraW5nLiBNb3JlIGRvaW5nLiIsICJJJ3ZlIHNlZW4gd29yc2UuIEJhcmVseS4iXSwKICAgICAgICApKSwKICAgICAgICBTdXJ2aXZvcihQZXJzb25hKAogICAgICAgICAgICBuYW1lPSJUaGVvIExpbmRxdmlzdCIsIHJvbGU9ImEgeW91bmcgcnVubmVyLCBmb3JtZXIgZ3JvY2VyeSBjbGVyayIsCiAgICAgICAgICAgIHBlcnNvbmFsaXR5PSJuZXJ2b3VzLCBraW5kLCBqb2tlcyB3aGVuIGhlIGlzIHNjYXJlZCIsCiAgICAgICAgICAgIHNwZWVjaF9zdHlsZT0icXVpY2ssIGFwb2xvZ2V0aWMsIHNlbGYtZGVwcmVjYXRpbmciLAogICAgICAgICAgICBrbm93bGVkZ2U9InRoZSBjYW1wJ3Mgc3RvcmVzIGFuZCB3aG8gYXRlIHdoYXQiLAogICAgICAgICAgICBmYWxsYmFja19saW5lcz1bIk9rYXkuIE9rYXkgb2theSBva2F5LiIsICJQbGVhc2UgdGVsbCBtZSB0aGF0IHdhcyB0aGUgd2luZC4iXSwKICAgICAgICApKSwKICAgIF0KCgpjbGFzcyBDYW1wOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHJuZz1Ob25lKToKICAgICAgICBzZWxmLnN1cnZpdm9ycyA9IHN0YXJ0ZXJfc3Vydml2b3JzKCkKICAgICAgICBzZWxmLmZvb2QgPSAzCiAgICAgICAgc2VsZi5tZWRzID0gMAogICAgICAgIHNlbGYuZGF5ID0gMQogICAgICAgIHNlbGYucm5nID0gcm5nIG9yIHJhbmRvbS5SYW5kb20oKQogICAgICAgIHNlbGYub3ZlciA9IEZhbHNlCiAgICAgICAgc2VsZi53b24gPSBGYWxzZQogICAgICAgIHNlbGYubG9nID0gW10KCiAgICAjIC0tLS0gc3RhdGUgLT4gdGV4dCAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICBkZWYgd29ybGRfc3RhdGUoc2VsZiwgczogU3Vydml2b3IsIG5pZ2h0X25vdGU6IHN0cikgLT4gc3RyOgogICAgICAgIHJldHVybiAoCiAgICAgICAgICAgIGYiRXZlbmluZyBhdCBhIHNtYWxsIHN1cnZpdm9yIGNhbXAgaW4gdGhlIHdvb2RzLCBkYXkge3NlbGYuZGF5fSBvZiAiCiAgICAgICAgICAgIGYidGhlIHNlY29uZCB3ZWVrIG9mIHRoZSBvdXRicmVhay4gRm9vZCBzdG9yZWQ6IHtzZWxmLmZvb2R9IG1lYWxzLiAiCiAgICAgICAgICAgIGYiTWVkaWNpbmUga2l0czoge3NlbGYubWVkc30uIHtuaWdodF9ub3RlfSAiCiAgICAgICAgICAgIGYie3MubmFtZX0gZmVlbHMge3MubW9yYWxlX3dvcmQoKX0gYW5kIGlzIHtzLmNvbmRpdGlvbl90ZXh0KCl9LiAiCiAgICAgICAgICAgIGYiVG9kYXkge3MubmFtZX0gc3BlbnQgdGhlIGRheSBvbiB7cy50b2RheX0uIgogICAgICAgICkKCiAgICAjIC0tLS0gZGF5IHJlc29sdXRpb24gLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICBkZWYgYXNzaWduKHNlbGYsIHM6IFN1cnZpdm9yLCB0YXNrOiBzdHIpOgogICAgICAgIGFzc2VydCB0YXNrIGluIFRBU0tTCiAgICAgICAgcy50YXNrID0gdGFzawoKICAgIGRlZiByZXNvbHZlX2RheShzZWxmLCBuaWdodF9ub3RlX2hvbGRlcjogbGlzdCk6CiAgICAgICAgZ3VhcmRlZCA9IGFueShzLnRhc2sgPT0gImd1YXJkIiBhbmQgcy5wcmVzZW50IGFuZCBzLmFsaXZlCiAgICAgICAgICAgICAgICAgICAgICBmb3IgcyBpbiBzZWxmLnN1cnZpdm9ycykKICAgICAgICBmb3IgcyBpbiBzZWxmLnN1cnZpdm9yczoKICAgICAgICAgICAgaWYgbm90IChzLmFsaXZlIGFuZCBzLnByZXNlbnQpOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgaWYgcy50YXNrID09ICJzY2F2ZW5nZSI6CiAgICAgICAgICAgICAgICBzLnRvZGF5ID0gInNjYXZlbmdpbmcgb3V0c2lkZSB0aGUgZmVuY2UiCiAgICAgICAgICAgICAgICBzZWxmLmZvb2QgKz0gMSBpZiBzLmhlYWx0aCAhPSAib2siIGVsc2UgMgogICAgICAgICAgICAgICAgaWYgc2VsZi5ybmcucmFuZG9tKCkgPCAwLjI1OgogICAgICAgICAgICAgICAgICAgIHNlbGYubWVkcyArPSAxCiAgICAgICAgICAgICAgICAgICAgc2VsZi5sb2cuYXBwZW5kKGYie3MubmFtZX0gZm91bmQgYSBtZWRpY2luZSBraXQuIikKICAgICAgICAgICAgICAgIHIgPSBzZWxmLnJuZy5yYW5kb20oKQogICAgICAgICAgICAgICAgaWYgcy5oZWFsdGggPT0gIm9rIiBhbmQgciA8IDAuMzA6CiAgICAgICAgICAgICAgICAgICAgcy5oZWFsdGggPSAiaHVydCIKICAgICAgICAgICAgICAgICAgICBzLm1vcmFsZSAtPSAxCiAgICAgICAgICAgICAgICAgICAgc2VsZi5sb2cuYXBwZW5kKGYie3MubmFtZX0gZ290IHNjcmF0Y2hlZCB1cCBvdXQgdGhlcmUuIikKICAgICAgICAgICAgICAgIGVsaWYgcy5oZWFsdGggPT0gImh1cnQiIGFuZCByIDwgMC41MDoKICAgICAgICAgICAgICAgICAgICBzLmhlYWx0aCA9ICJpbmZlY3RlZCIKICAgICAgICAgICAgICAgICAgICBzLm1vcmFsZSAtPSAyCiAgICAgICAgICAgICAgICAgICAgc2VsZi5sb2cuYXBwZW5kKGYie3MubmFtZX0ncyBzY3JhdGNoZXMgYXJlIGluZmVjdGVkIOKAlCBmZXZlciBzdGFydGluZy4iKQogICAgICAgICAgICBlbGlmIHMudGFzayA9PSAiZ3VhcmQiOgogICAgICAgICAgICAgICAgcy50b2RheSA9ICJzdGFuZGluZyBndWFyZCBhdCB0aGUgZmVuY2UiCiAgICAgICAgICAgIGVsaWYgcy50YXNrID09ICJyZXN0IjoKICAgICAgICAgICAgICAgIHMudG9kYXkgPSAicmVzdGluZyBhdCBjYW1wIgogICAgICAgICAgICAgICAgaWYgcy5oZWFsdGggPT0gImh1cnQiOgogICAgICAgICAgICAgICAgICAgIHMuaGVhbHRoID0gIm9rIgogICAgICAgICAgICAgICAgICAgIHNlbGYubG9nLmFwcGVuZChmIntzLm5hbWV9IHJlc3RlZCBhbmQgaXMgYmFjayBvbiB0aGVpciBmZWV0LiIpCiAgICAgICAgICAgICAgICBzLm1vcmFsZSA9IG1pbigxMCwgcy5tb3JhbGUgKyAyKQogICAgICAgICMgZXZlbmluZyBtZWFsCiAgICAgICAgaWYgc2VsZi5mb29kID49IDM6CiAgICAgICAgICAgIHNlbGYuZm9vZCAtPSAzCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc2VsZi5mb29kID0gMAogICAgICAgICAgICBmb3IgcyBpbiBzZWxmLnN1cnZpdm9yczoKICAgICAgICAgICAgICAgIHMubW9yYWxlIC09IDIKICAgICAgICAgICAgc2VsZi5sb2cuYXBwZW5kKCJOb3QgZW5vdWdoIGZvb2Qg4oCUIGV2ZXJ5b25lIHdlbnQgaHVuZ3J5LiIpCiAgICAgICAgaWYgc2VsZi5mb29kID49IDM6CiAgICAgICAgICAgIGZvciBzIGluIHNlbGYuc3Vydml2b3JzOgogICAgICAgICAgICAgICAgcy5tb3JhbGUgPSBtaW4oMTAsIHMubW9yYWxlICsgMSkKICAgICAgICAjIG5pZ2h0CiAgICAgICAgbmlnaHRfbm90ZSA9ICJUaGUgbmlnaHQgaXMgcXVpZXQuIgogICAgICAgIGlmIHNlbGYuZGF5ID09IDIgYW5kIG5vdCBndWFyZGVkOgogICAgICAgICAgICBzdG9sZW4gPSBtaW4oMiwgc2VsZi5mb29kKQogICAgICAgICAgICBzZWxmLmZvb2QgLT0gc3RvbGVuCiAgICAgICAgICAgIGZvciBzIGluIHNlbGYuc3Vydml2b3JzOgogICAgICAgICAgICAgICAgcy5tb3JhbGUgLT0gMQogICAgICAgICAgICBuaWdodF9ub3RlID0gIkluIHRoZSBuaWdodCwgYSBzdHJhbmdlciBzbGlwcGVkIGluIGFuZCBzdG9sZSBzdXBwbGllcy4iCiAgICAgICAgICAgIHNlbGYubG9nLmFwcGVuZCgiQSBzdHJhbmdlciByYWlkZWQgdGhlIHN0b3JlcyBpbiB0aGUgbmlnaHQg4oCUIG5vIG9uZSB3YXMgb24gZ3VhcmQuIikKICAgICAgICBlbGlmIHNlbGYuZGF5ID09IDI6CiAgICAgICAgICAgIG5pZ2h0X25vdGUgPSAiSW4gdGhlIG5pZ2h0LCBhIHN0cmFuZ2VyIGNyZXB0IHVwIHRvIHRoZSBmZW5jZSBidXQgd2FzIGRyaXZlbiBvZmYgYnkgdGhlIGd1YXJkLiIKICAgICAgICBuaWdodF9ub3RlX2hvbGRlci5hcHBlbmQobmlnaHRfbm90ZSkKICAgICAgICAjIGluZmVjdGlvbiBwcm9ncmVzc2lvbjogdW50cmVhdGVkICsgdW5yZXN0ZWQgPSBkZWF0aCBieSBtb3JuaW5nCiAgICAgICAgZm9yIHMgaW4gc2VsZi5zdXJ2aXZvcnM6CiAgICAgICAgICAgIGlmIHMuaGVhbHRoID09ICJpbmZlY3RlZCIgYW5kIHMudGFzayAhPSAicmVzdCI6CiAgICAgICAgICAgICAgICBzLmFsaXZlID0gRmFsc2UKICAgICAgICAgICAgICAgIHNlbGYubG9nLmFwcGVuZChmIntzLm5hbWV9IGRpZG4ndCBtYWtlIGl0IHRocm91Z2ggdGhlIG5pZ2h0LiBUaGUgZmV2ZXIgdG9vayB0aGVtLiIpCiAgICAgICAgIyBtb3JhbGUgY29sbGFwc2U6IHdhbGtzIG91dAogICAgICAgIGZvciBzIGluIHNlbGYuc3Vydml2b3JzOgogICAgICAgICAgICBpZiBzLmFsaXZlIGFuZCBzLm1vcmFsZSA8PSAwOgogICAgICAgICAgICAgICAgcy5wcmVzZW50ID0gRmFsc2UKICAgICAgICAgICAgICAgIHNlbGYubG9nLmFwcGVuZChmIntzLm5hbWV9IHdhbGtlZCBvdXQgaW50byB0aGUgZGFyayBhbmQgZGlkbid0IGNvbWUgYmFjay4iKQogICAgICAgIGlmIGFueShub3Qgcy5hbGl2ZSBvciBub3Qgcy5wcmVzZW50IGZvciBzIGluIHNlbGYuc3Vydml2b3JzKToKICAgICAgICAgICAgc2VsZi5vdmVyID0gVHJ1ZQogICAgICAgICAgICBzZWxmLndvbiA9IEZhbHNlCiAgICAgICAgZWxpZiBzZWxmLmRheSA+PSAzOgogICAgICAgICAgICBzZWxmLm92ZXIgPSBUcnVlCiAgICAgICAgICAgIHNlbGYud29uID0gVHJ1ZQoKICAgIGRlZiB1c2VfbWVkcyhzZWxmLCBzOiBTdXJ2aXZvcik6CiAgICAgICAgaWYgc2VsZi5tZWRzID4gMCBhbmQgcy5oZWFsdGggPT0gImluZmVjdGVkIjoKICAgICAgICAgICAgc2VsZi5tZWRzIC09IDEKICAgICAgICAgICAgcy5oZWFsdGggPSAiaHVydCIKICAgICAgICAgICAgc2VsZi5sb2cuYXBwZW5kKGYiWW91IGdhdmUge3MubmFtZX0gdGhlIG1lZGljaW5lLiBUaGUgZmV2ZXIgYnJlYWtzLiIpCiAgICAgICAgICAgIHJldHVybiBUcnVlCiAgICAgICAgcmV0dXJuIEZhbHNlCgoKIyAtLS0tIGludGVyYWN0aXZlIGxvb3AgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgpkZWYgcGxheShzcGVhaywgaW5wdXRfZm49aW5wdXQsIHByaW50X2ZuPXByaW50LCBzZWVkPU5vbmUpOgogICAgIiIic3BlYWsocGVyc29uYSwgd29ybGRfc3RhdGUpIC0+IHN0ciAob25lIGdlbmVyYXRlZCBsaW5lKS4iIiIKICAgIGNhbXAgPSBDYW1wKHJuZz1yYW5kb20uUmFuZG9tKHNlZWQpKQogICAgcHJpbnRfZm4oIkJBU0UgT0YgVEhSRUUg4oCUIHRocmVlIHN1cnZpdm9ycywgdGhyZWUgZGF5cy4gS2VlcCB0aGVtIGFsaXZlLCAiCiAgICAgICAgICAgICAiZmVkLCBhbmQgZnJvbSBnaXZpbmcgdXAuXG4iKQogICAgd2hpbGUgbm90IGNhbXAub3ZlcjoKICAgICAgICBwcmludF9mbihmIi0tLSBEQVkge2NhbXAuZGF5fSDigJQgZm9vZDoge2NhbXAuZm9vZH0sIG1lZHM6IHtjYW1wLm1lZHN9IC0tLSIpCiAgICAgICAgZm9yIHMgaW4gY2FtcC5zdXJ2aXZvcnM6CiAgICAgICAgICAgIHByaW50X2ZuKGYie3MubmFtZX06IG1vcmFsZSB7cy5tb3JhbGV9LzEwLCB7cy5oZWFsdGh9IikKICAgICAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgICAgIHQgPSBpbnB1dF9mbihmIiAgdGFzayBmb3Ige3MubmFtZS5zcGxpdCgpWzBdfSAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZiJbc2NhdmVuZ2UvZ3VhcmQvcmVzdF06ICIpLnN0cmlwKCkubG93ZXIoKQogICAgICAgICAgICAgICAgaWYgdCBpbiBUQVNLUzoKICAgICAgICAgICAgICAgICAgICBjYW1wLmFzc2lnbihzLCB0KQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICBwcmludF9mbigiICBwbGVhc2UgdHlwZSBzY2F2ZW5nZSwgZ3VhcmQsIG9yIHJlc3QiKQogICAgICAgICMgbWVkcyBkZWNpc2lvbiBmb3IgYW55b25lIGluZmVjdGVkCiAgICAgICAgZm9yIHMgaW4gY2FtcC5zdXJ2aXZvcnM6CiAgICAgICAgICAgIGlmIHMuaGVhbHRoID09ICJpbmZlY3RlZCIgYW5kIGNhbXAubWVkcyA+IDA6CiAgICAgICAgICAgICAgICBhID0gaW5wdXRfZm4oZiJ7cy5uYW1lfSBpcyBpbmZlY3RlZC4gVXNlIGEgbWVkaWNpbmUga2l0PyBbeS9uXTogIikKICAgICAgICAgICAgICAgIGlmIGEuc3RyaXAoKS5sb3dlcigpLnN0YXJ0c3dpdGgoInkiKToKICAgICAgICAgICAgICAgICAgICBjYW1wLnVzZV9tZWRzKHMpCiAgICAgICAgbm90ZSA9IFtdCiAgICAgICAgY2FtcC5yZXNvbHZlX2RheShub3RlKQogICAgICAgIGZvciBtc2cgaW4gY2FtcC5sb2c6CiAgICAgICAgICAgIHByaW50X2ZuKCIgKiAiICsgbXNnKQogICAgICAgIGNhbXAubG9nLmNsZWFyKCkKICAgICAgICBwcmludF9mbigiIikKICAgICAgICBmb3IgcyBpbiBjYW1wLnN1cnZpdm9yczoKICAgICAgICAgICAgaWYgcy5hbGl2ZSBhbmQgcy5wcmVzZW50OgogICAgICAgICAgICAgICAgbGluZSA9IHNwZWFrKHMucGVyc29uYSwgY2FtcC53b3JsZF9zdGF0ZShzLCBub3RlWzBdKSkKICAgICAgICAgICAgICAgIHByaW50X2ZuKGYne3MubmFtZX06ICJ7bGluZX0iJykKICAgICAgICBwcmludF9mbigiIikKICAgICAgICBjYW1wLmRheSArPSAxCiAgICBpZiBjYW1wLndvbjoKICAgICAgICBwcmludF9mbigiREFZIDQg4oCUIGRhd24uIEV2ZXJ5b25lIGlzIHN0aWxsIGhlcmUuIFRoZSBjYW1wIGhvbGRzLiBZT1UgV0lOLiIpCiAgICBlbHNlOgogICAgICAgIHByaW50X2ZuKCJUaGUgY2FtcCBpcyBicm9rZW4uIEdBTUUgT1ZFUi4iKQogICAgcmV0dXJuIGNhbXAud29uCg==',
}
for rel, b64 in FILES.items():
    p = pathlib.Path(rel)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(base64.b64decode(b64))
import npc_engine
print("npc_engine version:", npc_engine.__version__, " (must say 0.3.0)")

## 3. Play

Uses your fine-tuned `npc_adapter` automatically if it exists in this runtime; otherwise the base model plays.

In [ ]:
import os
from npc_engine import NPCEngine
from npc_engine.games.base_of_three import play, BARK_PROMPT

adapter = 'npc_adapter' if os.path.exists('npc_adapter/adapter_config.json') else None
print('engine:', 'fine-tuned adapter' if adapter else 'base model')
eng = NPCEngine(load_in_8bit=True, adapter_path=adapter)

def speak(persona, world):
    r = eng.respond(persona, world, BARK_PROMPT, steps=16,
                    n_candidates=4, max_new_tokens=48)
    return r.line

play(speak)